# 01. 埋め込みモデルのアンサンブル重み最適化

TF-IDF と Hugging Face の埋め込みモデル3つの類似度を加重平均し、検証データの平均順位が最小になる重みをグリッドサーチで決定する。

In [2]:
import gc
from itertools import product

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sudachipy import dictionary, tokenizer
from tqdm.notebook import tqdm

In [3]:
# 使用する埋め込みモデル
MODEL_NAMES = [
    "Qwen/Qwen3-Embedding-0.6B",
    "cl-nagoya/ruri-v3-310m",
    "stsb-xlm-r-multilingual",
]

## データ読み込み

In [4]:
base_df = pd.read_csv("../dataset/base_stories.tsv", sep="\t")
practice_df = pd.read_csv("../dataset/fiction_stories_practice.tsv", sep="\t")

base_stories = base_df["story"].tolist()
practice_stories = practice_df["story"].tolist()

print(f"Base stories: {len(base_stories)}")
print(f"Practice stories: {len(practice_stories)}")

Base stories: 50
Practice stories: 20


## テキスト処理（Sudachipy）

TF-IDF用に形態素解析を行い、名詞・動詞・形容詞を抽出する。

In [5]:
class TextProcessor:
    """Sudachipyを使用したテキスト処理"""

    def __init__(self):
        self.tokenizer_obj = dictionary.Dictionary().create()
        self.mode = tokenizer.Tokenizer.SplitMode.C

    def tokenize(self, text: str) -> str:
        tokens = self.tokenizer_obj.tokenize(text, self.mode)
        words = [
            token.normalized_form()
            for token in tokens
            if token.part_of_speech()[0] in ["名詞", "動詞", "形容詞"]
        ]
        return " ".join(words)


text_processor = TextProcessor()

## 各モデルの類似度行列を計算

3つの埋め込みモデルとTF-IDFそれぞれで `(practice, base)` の類似度行列を算出し、MinMaxスケーリングで正規化する。

In [6]:
similarities = {}

# SentenceTransformer モデル
for model_name in MODEL_NAMES:
    print(f"Processing: {model_name}")
    model = SentenceTransformer(model_name)

    base_emb = model.encode(base_stories, show_progress_bar=True)
    practice_emb = model.encode(practice_stories, show_progress_bar=True)
    sim = cosine_similarity(practice_emb, base_emb)

    key = model_name.split("/")[-1]
    similarities[key] = sim
    print(f"  Shape: {sim.shape}, Range: [{sim.min():.4f}, {sim.max():.4f}]")

    del model
    gc.collect()

Processing: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

/home/sakamoto/competition/signate/movie_story/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/stsb-xlm-r-multilingual
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  Shape: (20, 50), Range: [0.1093, 0.8466]


In [7]:
# TF-IDF
print("Processing: TF-IDF")
base_tokenized = [text_processor.tokenize(t) for t in tqdm(base_stories, desc="Base tokenize")]
practice_tokenized = [text_processor.tokenize(t) for t in tqdm(practice_stories, desc="Practice tokenize")]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(base_tokenized + practice_tokenized)
base_tfidf = tfidf_matrix[:len(base_tokenized)]
practice_tfidf = tfidf_matrix[len(base_tokenized):]

sim_tfidf = cosine_similarity(practice_tfidf, base_tfidf)
similarities["tfidf"] = sim_tfidf
print(f"  Shape: {sim_tfidf.shape}, Range: [{sim_tfidf.min():.4f}, {sim_tfidf.max():.4f}]")

Processing: TF-IDF


Base tokenize:   0%|          | 0/50 [00:00<?, ?it/s]

Practice tokenize:   0%|          | 0/20 [00:00<?, ?it/s]

  Shape: (20, 50), Range: [0.0175, 0.4423]


In [8]:
# MinMaxスケーリング
scaled = {}
for key, sim in similarities.items():
    scaler = MinMaxScaler()
    flat = sim.flatten().reshape(-1, 1)
    scaled[key] = scaler.fit_transform(flat).reshape(sim.shape)

print("Scaled keys:", list(scaled.keys()))

Scaled keys: ['Qwen3-Embedding-0.6B', 'ruri-v3-310m', 'stsb-xlm-r-multilingual', 'tfidf']


## グリッドサーチによる重み最適化

検証データ（practice）の正解ペア `(id_a, id_b)` に対して、加重平均した類似度行列での平均順位を評価指標とする。

重みを0.0〜1.0の範囲で0.1刻みに探索し、合計が1.0になる組み合わせのみを対象とする。

In [9]:
def evaluate_weights(weights: dict, scaled: dict, practice_df: pd.DataFrame, base_df: pd.DataFrame) -> float:
    """加重平均した類似度行列で、正解ペアの平均順位を算出する"""
    combined = np.zeros_like(list(scaled.values())[0])
    for key, sim in scaled.items():
        combined += weights[key] * sim

    total_rank = 0
    for i in range(len(practice_df)):
        actual_a = int(practice_df.iloc[i]["id_a"])
        actual_b = int(practice_df.iloc[i]["id_b"])

        sim_scores = combined[i]
        ranked_ids = base_df["id"].values[np.argsort(sim_scores)[::-1]]

        rank_a = np.where(ranked_ids == actual_a)[0][0] + 1
        rank_b = np.where(ranked_ids == actual_b)[0][0] + 1
        total_rank += rank_a + rank_b

    return total_rank / (len(practice_df) * 2)

In [10]:
# グリッドサーチ
keys = list(scaled.keys())
step = 0.1
values = np.arange(0.0, 1.0 + step, step).round(1)

best_score = float("inf")
best_weights = None
results = []

for combo in product(values, repeat=len(keys)):
    if abs(sum(combo) - 1.0) > 1e-6:
        continue

    weights = dict(zip(keys, combo))
    score = evaluate_weights(weights, scaled, practice_df, base_df)
    results.append({**weights, "mean_rank": score})

    if score < best_score:
        best_score = score
        best_weights = weights

print(f"探索した組み合わせ数: {len(results)}")
print(f"\nベストスコア（平均順位）: {best_score:.2f}")
print(f"ベスト重み:")
for k, v in best_weights.items():
    print(f"  {k}: {v}")

探索した組み合わせ数: 286

ベストスコア（平均順位）: 2.42
ベスト重み:
  Qwen3-Embedding-0.6B: 0.2
  ruri-v3-310m: 0.2
  stsb-xlm-r-multilingual: 0.1
  tfidf: 0.5


In [11]:
# 結果一覧（上位10件）
results_df = pd.DataFrame(results).sort_values("mean_rank")
results_df.head(10)

,Qwen3-Embedding-0.6B,ruri-v3-310m,stsb-xlm-r-multilingual,tfidf,mean_rank
139,0.2,0.2,0.1,0.5,2.425
181,0.3,0.2,0.0,0.5,2.425
175,0.3,0.1,0.1,0.5,2.425
146,0.2,0.3,0.1,0.4,2.450
182,0.3,0.2,0.1,0.4,2.450
152,0.2,0.4,0.1,0.3,2.450
145,0.2,0.3,0.0,0.5,2.450
188,0.3,0.3,0.1,0.3,2.450
187,0.3,0.3,0.0,0.4,2.450
193,0.3,0.4,0.1,0.2,2.450
